NovaCart-Generative AI Customer Support Assistant

Objective:
Build a Generative AI-powered customer support assistant that answers customer questions using a predefined knowledge base and provide safe fallback responses when information is unavailable.

In [2]:
knowledge_base = {
    "orders": {
        "order_status": "Customers can check their order status from the 'My Orders' section of their NovaCart account.",
        "processing": "Orders are normally processed within 1–2 business days.",
        "address_change": "Address changes are possible only before the order has been shipped.",
        "cancellation": "Orders can be cancelled before they are shipped."
    },

    "shipping": {
        "standard_delivery": "Standard delivery normally takes 3–5 business days after order processing.",
        "express_delivery": "Express delivery is available for eligible locations and normally takes 1–2 business days.",
        "tracking": "Customers can track their order through the 'My Orders' section after the order has been shipped.",
        "delayed_order": "Customers should first check the tracking information. If the order remains delayed, they can contact NovaCart support."
    },

    "returns": {
        "return_policy": "Eligible products can be returned within 7 days of delivery.",
        "condition": "Products must generally be unused, undamaged, and returned with the original packaging and accessories.",
        "eligibility": "Certain products may be excluded from the return policy.",
        "request_return": "Customers can request a return through the 'My Orders' section."
    },

    "refunds": {
        "refund_time": "Refunds are normally processed within 5–7 business days after the returned product is received and approved.",
        "refund_method": "Refunds are generally issued to the original payment method.",
        "missing_refund": "Customers should verify the expected processing period and contact customer support if the refund is still missing."
    },

    "payments": {
        "payment_methods": "NovaCart accepts major debit cards, credit cards, UPI, and other payment methods available during checkout.",
        "payment_failed": "Customers should verify their payment details and try again. If the problem continues, they should contact their bank or NovaCart support.",
        "charged_without_order": "Customers should not make another payment immediately. They should contact NovaCart support with their transaction details."
    },

    "account_support": {
        "forgot_password": "Customers can use the 'Forgot Password' option on the NovaCart login page.",
        "contact_support": "Customers can contact NovaCart support through the support section of the website.",
        "human_agent": "Customers can request human assistance when the AI assistant cannot resolve their issue."
    }
}

print("Knowledge base loaded successfully!")

Knowledge base loaded successfully!


In [3]:
documents = []

for category, questions in knowledge_base.items():
    for topic, answer in questions.items():
        documents.append({
            "category": category,
            "topic": topic,
            "content": answer
        })

print(f"Created {len(documents)} knowledge documents.")

Created 21 knowledge documents.


In [4]:
for document in documents:
    print(document)

{'category': 'orders', 'topic': 'order_status', 'content': "Customers can check their order status from the 'My Orders' section of their NovaCart account."}
{'category': 'orders', 'topic': 'processing', 'content': 'Orders are normally processed within 1–2 business days.'}
{'category': 'orders', 'topic': 'address_change', 'content': 'Address changes are possible only before the order has been shipped.'}
{'category': 'orders', 'topic': 'cancellation', 'content': 'Orders can be cancelled before they are shipped.'}
{'category': 'shipping', 'topic': 'standard_delivery', 'content': 'Standard delivery normally takes 3–5 business days after order processing.'}
{'category': 'shipping', 'topic': 'express_delivery', 'content': 'Express delivery is available for eligible locations and normally takes 1–2 business days.'}
{'category': 'shipping', 'topic': 'tracking', 'content': "Customers can track their order through the 'My Orders' section after the order has been shipped."}
{'category': 'shipping

In [5]:
def search_knowledge_base(query):
    query_words = query.lower().split()
    results = []

    for document in documents:
        text = (
            document["category"] + " " +
            document["topic"] + " " +
            document["content"]
        ).lower()

        score = sum(word in text for word in query_words)

        if score > 0:
            results.append((score, document))

    results.sort(reverse=True, key=lambda x: x[0])

    return results

In [6]:
query = "How long does delivery take?"

results = search_knowledge_base(query)

for score, document in results[:3]:
    print("Score:", score)
    print("Category:", document["category"])
    print("Topic:", document["topic"])
    print("Information:", document["content"])
    print("-" * 50)

Score: 1
Category: shipping
Topic: standard_delivery
Information: Standard delivery normally takes 3–5 business days after order processing.
--------------------------------------------------
Score: 1
Category: shipping
Topic: express_delivery
Information: Express delivery is available for eligible locations and normally takes 1–2 business days.
--------------------------------------------------
Score: 1
Category: returns
Topic: return_policy
Information: Eligible products can be returned within 7 days of delivery.
--------------------------------------------------


In [7]:
!pip install -q sentence-transformers

In [8]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

print("Embedding model loaded successfully!")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded successfully!


In [9]:
document_texts = [
    document["category"] + " " +
    document["topic"] + " " +
    document["content"]
    for document in documents
]

document_embeddings = embedding_model.encode(document_texts)

print("Created embeddings for", len(document_embeddings), "documents.")

Created embeddings for 21 documents.


In [13]:
from sklearn.metrics.pairwise import cosine_similarity

def semantic_search(query, top_k=3):
    query_embedding = embedding_model.encode([query])

    similarities = cosine_similarity(
        query_embedding,
        document_embeddings
    )[0]

    top_indices = similarities.argsort()[-top_k:][::-1]

    results = []

    for index in top_indices:
        results.append({
            "score": similarities[index],
            "document": documents[index]
        })

    return results

In [14]:
results = semantic_search("How can I reset my password?", top_k=3)

for i, result in enumerate(results, 1):
    print(f"\nResult {i}")
    print("Score:", result["score"])
    print("Document:", result["document"])


Result 1
Score: 0.48708448
Document: {'category': 'account_support', 'topic': 'forgot_password', 'content': "Customers can use the 'Forgot Password' option on the NovaCart login page."}

Result 2
Score: 0.24618603
Document: {'category': 'account_support', 'topic': 'contact_support', 'content': 'Customers can contact NovaCart support through the support section of the website.'}

Result 3
Score: 0.15232165
Document: {'category': 'payments', 'topic': 'payment_failed', 'content': 'Customers should verify their payment details and try again. If the problem continues, they should contact their bank or NovaCart support.'}


In [15]:
def generate_answer(query):
    results = semantic_search(query, top_k=3)

    best_result = results[0]

    answer = best_result["document"]["content"]

    return answer

In [16]:
answer = generate_answer("How can I reset my password?")

print("Customer:", "How can I reset my password?")
print("Assistant:", answer)

Customer: How can I reset my password?
Assistant: Customers can use the 'Forgot Password' option on the NovaCart login page.


In [17]:
while True:
    user_query = input("\nCustomer: ")

    if user_query.lower() in ["exit", "quit", "bye"]:
        print("Assistant: Thank you for contacting NovaCart Support!")
        break

    answer = generate_answer(user_query)

    print("Assistant:", answer)


Customer: i forgot my password
Assistant: Customers can use the 'Forgot Password' option on the NovaCart login page.

Customer: exit
Assistant: Thank you for contacting NovaCart Support!


In [18]:
def generate_answer(query):
    results = semantic_search(query, top_k=3)

    best_document = results[0]["document"]

    return (
        f"Based on NovaCart's support information: "
        f"{best_document['content']}"
    )

In [19]:
questions = [
    "I forgot my password. What should I do?",
    "My payment failed. What can I do?",
    "How can I contact NovaCart support?"
]

for question in questions:
    print("\nCustomer:", question)
    print("Assistant:", generate_answer(question))


Customer: I forgot my password. What should I do?
Assistant: Based on NovaCart's support information: Customers can use the 'Forgot Password' option on the NovaCart login page.

Customer: My payment failed. What can I do?
Assistant: Based on NovaCart's support information: Customers should verify their payment details and try again. If the problem continues, they should contact their bank or NovaCart support.

Customer: How can I contact NovaCart support?
Assistant: Based on NovaCart's support information: Customers can contact NovaCart support through the support section of the website.


In [20]:
import transformers
print("Transformers version:", transformers.__version__)

Transformers version: 5.15.0


In [21]:
from transformers import pipeline

generator = pipeline(
    "text-generation",
    model="Qwen/Qwen2.5-0.5B-Instruct"
)

print("Generative AI model loaded successfully!")

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

Generative AI model loaded successfully!


In [22]:
def generate_rag_answer(query):
    results = semantic_search(query, top_k=3)

    context = "\n".join(
        [result["document"]["content"] for result in results]
    )

    prompt = f"""You are NovaCart's customer support assistant.

Use ONLY the information provided in the knowledge base below to answer the customer's question.

Knowledge base:
{context}

Customer question:
{query}

Give a short, clear and helpful answer. If the knowledge base does not contain enough information, say that you don't have enough information and recommend contacting NovaCart support.

Assistant:"""

    response = generator(
        prompt,
        max_new_tokens=100,
        do_sample=False
    )

    generated_text = response[0]["generated_text"]

    answer = generated_text.split("Assistant:")[-1].strip()

    return answer

In [23]:
question = "I forgot my password. What should I do?"

answer = generate_rag_answer(question)

print("Customer:", question)
print("Assistant:", answer)

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


Customer: I forgot my password. What should I do?
Assistant: To reset your password, follow these steps:

1. Go to the NovaCart login page.
2. Click on the "Forgot Password" link.
3. Enter your email address associated with your account.
4. Follow the instructions to complete the password reset process.

If you still cannot find the password reset link, please check your spam folder for any new emails from NovaCart. Alternatively, you can also try using a different email address associated with your account. If you're still unable to reset your


In [26]:
def generate_rag_answer(query):
    results = semantic_search(query, top_k=1)

    context = results[0]["document"]["content"]

    prompt = f"""You are a customer support assistant.

Answer the customer's question using ONLY the exact information in the knowledge base.

Knowledge base:
{context}

Customer question:
{query}

Rules:
- Do not add any information.
- Do not invent steps or instructions.
- Do not mention information that is not in the knowledge base.
- If the knowledge base answers the question, give only that answer in one or two sentences.
- If it does not answer the question, reply exactly:
"I don't have enough information to answer that. Please contact NovaCart support."

Answer:"""

    response = generator(
        prompt,
        max_new_tokens=40,
        do_sample=False,
        return_full_text=False,
        generation_config=None
    )

    answer = response[0]["generated_text"].strip()

    return answer

In [27]:
question = "I forgot my password. What should I do?"

answer = generate_rag_answer(question)

print("Customer:", question)
print("Assistant:", answer)

Customer: I forgot my password. What should I do?
Assistant: "Please enter your email address and click the 'Forgot Password' button on the NovaCart login page."


In [62]:
def grounded_answer(query):
    results = semantic_search(query, top_k=3)

    best_result = results[0]
    second_result = results[1]

    best_score = best_result["score"]
    second_score = second_result["score"]

    score_gap = best_score - second_score

    # Relevance check
    if best_score < 0.40 or score_gap < 0.02:
        return "I don't have enough information to answer that. Please contact NovaCart support."

    return best_result["document"]["content"]

In [63]:
test_questions = [
    "Where can I see the status of my shipped order?",
    "I changed my mind. Can I stop my order?",
    "Can I get a discount?"
]

for question in test_questions:
    print("\nCustomer:", question)
    print("Assistant:", professional_answer(question))


Customer: Where can I see the status of my shipped order?
Assistant: I'm happy to help. Customers can check their order status from the 'My Orders' section of their NovaCart account.

Customer: I changed my mind. Can I stop my order?
Assistant: I'm happy to help. Orders can be cancelled before they are shipped.

Customer: Can I get a discount?
Assistant: I don't have enough information to answer that. Please contact NovaCart support.


In [51]:
test_questions = [
    "I forgot my password.",
    "My payment failed.",
    "Can I get a discount?"
]

for question in test_questions:
    print("\nCustomer:", question)
    print("Assistant:", professional_answer(question))


Customer: I forgot my password.
Assistant: I'm happy to help. Customers can use the 'Forgot Password' option on the NovaCart login page.

Customer: My payment failed.
Assistant: I'm happy to help. Customers should verify their payment details and try again. If the problem continues, they should contact their bank or NovaCart support.

Customer: Can I get a discount?
Assistant: I don't have enough information to answer that. Please contact NovaCart support.


In [33]:
test_questions = [
    "How can I contact NovaCart support?",
    "My payment failed. What should I do?",
    "Can I get a discount on my next order?"
]

for question in test_questions:
    print("\nCustomer:", question)
    print("Assistant:", grounded_answer(question))


Customer: How can I contact NovaCart support?
Assistant: Customers can contact NovaCart support through the support section of the website.

Customer: My payment failed. What should I do?
Assistant: Customers should verify their payment details and try again. If the problem continues, they should contact their bank or NovaCart support.

Customer: Can I get a discount on my next order?
Assistant: I don't have enough information to answer that. Please contact NovaCart support.


In [29]:
question = "I forgot my password?"

print("Customer:", question)
print("Assistant:", grounded_answer(question))

Customer: I forgot my password?
Assistant: Customers can use the 'Forgot Password' option on the NovaCart login page.


In [30]:
test_questions = [
    "How can I contact NovaCart support?",
    "My payment failed. What should I do?",
    "Can I get a discount on my next order?"
]

for question in test_questions:
    print("\nCustomer:", question)
    print("Assistant:", grounded_answer(question))


Customer: How can I contact NovaCart support?
Assistant: Customers can contact NovaCart support through the support section of the website.

Customer: My payment failed. What should I do?
Assistant: Customers should verify their payment details and try again. If the problem continues, they should contact their bank or NovaCart support.

Customer: Can I get a discount on my next order?
Assistant: Customers can track their order through the 'My Orders' section after the order has been shipped.


In [31]:
question = "Can I get a discount on my next order?"

results = semantic_search(question, top_k=3)

for i, result in enumerate(results, 1):
    print(f"Result {i}")
    print("Score:", result["score"])
    print("Document:", result["document"])
    print()

Result 1
Score: 0.42847618
Document: {'category': 'shipping', 'topic': 'tracking', 'content': "Customers can track their order through the 'My Orders' section after the order has been shipped."}

Result 2
Score: 0.42033646
Document: {'category': 'orders', 'topic': 'cancellation', 'content': 'Orders can be cancelled before they are shipped.'}

Result 3
Score: 0.39724708
Document: {'category': 'payments', 'topic': 'charged_without_order', 'content': 'Customers should not make another payment immediately. They should contact NovaCart support with their transaction details.'}



In [11]:
query = "When will my package arrive?"

results = semantic_search(query)

for result in results:
    print("Similarity:", round(result["score"], 3))
    print("Category:", result["document"]["category"])
    print("Topic:", result["document"]["topic"])
    print("Information:", result["document"]["content"])
    print("-" * 50)

Similarity: 0.486
Category: shipping
Topic: standard_delivery
Information: Standard delivery normally takes 3–5 business days after order processing.
--------------------------------------------------
Similarity: 0.474
Category: shipping
Topic: express_delivery
Information: Express delivery is available for eligible locations and normally takes 1–2 business days.
--------------------------------------------------
Similarity: 0.433
Category: orders
Topic: processing
Information: Orders are normally processed within 1–2 business days.
--------------------------------------------------


In [12]:
!pip install -q transformers accelerate

In [34]:
import gradio as gr

print("Gradio version:", gr.__version__)

Gradio version: 6.24.0


In [37]:
import gradio as gr

def chat_with_support(message, history):
    if not message.strip():
        return "", history

    answer = grounded_answer(message)

    history = history + [
        {"role": "user", "content": message},
        {"role": "assistant", "content": answer}
    ]

    return "", history


demo = gr.ChatInterface(
    fn=chat_with_support,
    title="NovaCart Customer Support Assistant",
    description="Ask a question about your NovaCart account, orders, payments, or support.",
    textbox=gr.Textbox(
        placeholder="Type your question here...",
        container=True
    )
)

demo.launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://4086f46b3e7d245786.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/gradio/queueing.py", line 870, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<5 lines>...
    )
    ^
  File "/usr/local/lib/python3.13/dist-packages/gradio/route_utils.py", line 409, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<12 lines>...
    )
    ^
  File "/usr/local/lib/python3.13/dist-packages/gradio/blocks.py", line 2317, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<9 lines>...
    )
    ^
  File "/usr/local/lib/python3.13/dist-packages/gradio/blocks.py", line 1682, in call_function
    prediction = await fn(*processed_input)
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.13/dist-packages/gradio/utils.py", line 1081, in async_wrapper
    response = await 

Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7861 <> https://4086f46b3e7d245786.gradio.live


In [40]:
def chat_with_support(message, history):
    if not message.strip():
        return "", history

    answer = grounded_answer(message)

    history = history or []

    history.append({
        "role": "user",
        "content": message
    })

    history.append({
        "role": "assistant",
        "content": answer
    })

    return "", history


demo = gr.ChatInterface(
    fn=chat_with_support,
    title="NovaCart Customer Support Assistant",
    description="Ask a question about your NovaCart account, orders, payments, or support.",
    textbox=gr.Textbox(
        placeholder="Type your question here...",
        container=True
    ),
    type="messages"
)

demo.launch(share=True, debug=True)

TypeError: ChatInterface.__init__() got an unexpected keyword argument 'type'

In [41]:
def chat_with_support(message, history):
    if not message.strip():
        return ""

    answer = grounded_answer(message)

    return answer


demo = gr.ChatInterface(
    fn=chat_with_support,
    title="NovaCart Customer Support Assistant",
    description="Ask a question about your NovaCart account, orders, payments, or support.",
    textbox=gr.Textbox(
        placeholder="Type your question here...",
        container=True
    )
)

demo.launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://79159ad5a1dc1d2b07.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7861 <> https://79159ad5a1dc1d2b07.gradio.live


In [42]:
chat_history = []

def chat_with_support(message, history):
    global chat_history

    if not message.strip():
        return ""

    chat_history.append(message)

    answer = grounded_answer(message)

    chat_history.append(answer)

    return answer

In [43]:
def chat_with_support(message, history):
    if not message.strip():
        return ""

    # Build context from previous conversation
    conversation = ""

    if history:
        for item in history:
            if isinstance(item, dict):
                role = item.get("role", "")
                content = item.get("content", "")
                conversation += f"{role}: {content}\n"

    # Use the current question for retrieval
    answer = grounded_answer(message)

    return answer

In [44]:
demo = gr.ChatInterface(
    fn=chat_with_support,
    title="NovaCart Customer Support Assistant",
    description="Ask a question about your NovaCart account, orders, payments, or support.",
    textbox=gr.Textbox(
        placeholder="Type your question here...",
        container=True
    )
)

demo.launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://e607fb8d89e65ca4fb.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7861 <> https://e607fb8d89e65ca4fb.gradio.live


In [45]:
def chat_with_support(message, history):
    if not message.strip():
        return ""

    # If there is previous conversation, combine it with the new question
    search_query = message

    if history:
        previous_messages = []

        for item in history:
            if isinstance(item, dict):
                role = item.get("role", "")
                content = item.get("content", "")

                if content:
                    previous_messages.append(f"{role}: {content}")

        if previous_messages:
            search_query = "\n".join(previous_messages[-4:]) + "\nuser: " + message

    answer = grounded_answer(search_query)

    return answer

In [46]:
demo = gr.ChatInterface(
    fn=chat_with_support,
    title="NovaCart Customer Support Assistant",
    description="Ask a question about your NovaCart account, orders, payments, or support.",
    textbox=gr.Textbox(
        placeholder="Type your question here...",
        container=True
    )
)

demo.launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://297f26b9d6fcd82d09.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7861 <> https://297f26b9d6fcd82d09.gradio.live


In [47]:
def professional_answer(query):
    answer = grounded_answer(query)

    if answer.startswith("I don't have enough information"):
        return answer

    return f"I'm happy to help. {answer}"

In [48]:
test_questions = [
    "I forgot my password.",
    "My payment failed.",
    "Can I get a discount?"
]

for question in test_questions:
    print("\nCustomer:", question)
    print("Assistant:", professional_answer(question))


Customer: I forgot my password.
Assistant: I don't have enough information to answer that. Please contact NovaCart support.

Customer: My payment failed.
Assistant: I'm happy to help. Customers should verify their payment details and try again. If the problem continues, they should contact their bank or NovaCart support.

Customer: Can I get a discount?
Assistant: I don't have enough information to answer that. Please contact NovaCart support.


In [49]:
question = "I forgot my password."

results = semantic_search(question, top_k=3)

for i, result in enumerate(results, 1):
    print(f"Result {i}")
    print("Score:", result["score"])
    print("Topic:", result["document"]["topic"])
    print("Content:", result["document"]["content"])
    print()

Result 1
Score: 0.45802468
Topic: forgot_password
Content: Customers can use the 'Forgot Password' option on the NovaCart login page.

Result 2
Score: 0.22876948
Topic: contact_support
Content: Customers can contact NovaCart support through the support section of the website.

Result 3
Score: 0.15384023
Topic: request_return
Content: Customers can request a return through the 'My Orders' section.



In [52]:
import gradio as gr

demo = gr.ChatInterface(
    fn=chat_with_support,
    title="🛒 NovaCart Customer Support Assistant",
    description=(
        "Welcome to NovaCart Support! "
        "Ask questions about your account, orders, payments, or other support topics."
    ),
    examples=[
        "I forgot my password.",
        "My payment failed. What should I do?",
        "How can I contact NovaCart support?",
        "Can I get a discount on my next order?"
    ],
    textbox=gr.Textbox(
        placeholder="Type your question here...",
        container=True
    )
)

demo.launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://37d8fcbd1bd4d8a7ef.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7861 <> https://37d8fcbd1bd4d8a7ef.gradio.live


In [53]:
def chat_with_support(message, history):
    if not message.strip():
        return ""

    # Retrieve knowledge using ONLY the current customer question
    answer = professional_answer(message)

    return answer

In [54]:
demo = gr.ChatInterface(
    fn=chat_with_support,
    title="🛒 NovaCart Customer Support Assistant",
    description=(
        "Welcome to NovaCart Support! "
        "Ask questions about your account, orders, payments, or other support topics."
    ),
    examples=[
        "I forgot my password.",
        "My payment failed. What should I do?",
        "How can I contact NovaCart support?",
        "Can I get a discount?"
    ],
    textbox=gr.Textbox(
        placeholder="Type your question here...",
        container=True
    )
)

demo.launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://89b3e4f734add556b2.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7861 <> https://89b3e4f734add556b2.gradio.live


In [55]:
test_cases = [
    "I forgot my password.",
    "How do I reset my password?",
    "My payment failed.",
    "What should I do if my payment doesn't go through?",
    "How can I contact support?",
    "Where can I contact NovaCart?",
    "Can I get a discount?",
    "Do you offer discounts?",
    "How do I track my order?",
    "Can I cancel my order?"
]

for question in test_cases:
    answer = professional_answer(question)

    print("Customer:", question)
    print("Assistant:", answer)
    print("-" * 80)

Customer: I forgot my password.
Assistant: I'm happy to help. Customers can use the 'Forgot Password' option on the NovaCart login page.
--------------------------------------------------------------------------------
Customer: How do I reset my password?
Assistant: I'm happy to help. Customers can use the 'Forgot Password' option on the NovaCart login page.
--------------------------------------------------------------------------------
Customer: My payment failed.
Assistant: I'm happy to help. Customers should verify their payment details and try again. If the problem continues, they should contact their bank or NovaCart support.
--------------------------------------------------------------------------------
Customer: What should I do if my payment doesn't go through?
Assistant: I'm happy to help. Customers should verify their payment details and try again. If the problem continues, they should contact their bank or NovaCart support.
-------------------------------------------------

In [56]:
def get_answer_with_source(query):
    results = semantic_search(query, top_k=3)

    best_result = results[0]
    second_result = results[1]

    best_score = best_result["score"]
    second_score = second_result["score"]
    score_gap = best_score - second_score

    # Relevance check
    if best_score < 0.40 or score_gap < 0.05:
        return {
            "answer": "I don't have enough information to answer that. Please contact NovaCart support.",
            "topic": "No relevant information",
            "score": best_score
        }

    return {
        "answer": f"I'm happy to help. {best_result['document']['content']}",
        "topic": best_result["document"]["topic"],
        "score": best_score
    }

In [57]:
result = get_answer_with_source("My payment failed.")

print("Answer:", result["answer"])
print("Knowledge Topic:", result["topic"])
print("Similarity Score:", result["score"])

Answer: I'm happy to help. Customers should verify their payment details and try again. If the problem continues, they should contact their bank or NovaCart support.
Knowledge Topic: payment_failed
Similarity Score: 0.71273214


In [58]:
test_questions = [
    "I forgot my password.",
    "My payment failed.",
    "Can I get a discount?"
]

for question in test_questions:
    result = get_answer_with_source(question)

    print("\nCustomer:", question)
    print("Answer:", result["answer"])
    print("Topic:", result["topic"])
    print("Score:", round(result["score"], 3))
    print("-" * 70)


Customer: I forgot my password.
Answer: I'm happy to help. Customers can use the 'Forgot Password' option on the NovaCart login page.
Topic: forgot_password
Score: 0.458
----------------------------------------------------------------------

Customer: My payment failed.
Answer: I'm happy to help. Customers should verify their payment details and try again. If the problem continues, they should contact their bank or NovaCart support.
Topic: payment_failed
Score: 0.713
----------------------------------------------------------------------

Customer: Can I get a discount?
Answer: I don't have enough information to answer that. Please contact NovaCart support.
Topic: No relevant information
Score: 0.28
----------------------------------------------------------------------


In [59]:
def chat_with_support(message, history):
    if not message.strip():
        return ""

    result = get_answer_with_source(message)

    answer = result["answer"]
    topic = result["topic"]

    if topic != "No relevant information":
        answer += f"\n\n*Support topic: {topic.replace('_', ' ').title()}*"

    return answer

In [60]:
demo = gr.ChatInterface(
    fn=chat_with_support,
    title="🛒 NovaCart Customer Support Assistant",
    description=(
        "Welcome to NovaCart Support! "
        "Ask questions about your account, orders, payments, or other support topics."
    ),
    examples=[
        "I forgot my password.",
        "My payment failed. What should I do?",
        "How can I contact NovaCart support?",
        "Can I get a discount?"
    ],
    textbox=gr.Textbox(
        placeholder="Type your question here...",
        container=True
    )
)

demo.launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://e8061044e954117d56.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7861 <> https://e8061044e954117d56.gradio.live


In [61]:
questions = [
    "Where can I see the status of my shipped order?",
    "I changed my mind. Can I stop my order?"
]

for question in questions:
    print("\nQUESTION:", question)

    results = semantic_search(question, top_k=3)

    for i, result in enumerate(results, 1):
        print(f"Result {i}")
        print("Score:", round(result["score"], 3))
        print("Topic:", result["document"]["topic"])
        print("Content:", result["document"]["content"])


QUESTION: Where can I see the status of my shipped order?
Result 1
Score: 0.61
Topic: order_status
Content: Customers can check their order status from the 'My Orders' section of their NovaCart account.
Result 2
Score: 0.588
Topic: tracking
Content: Customers can track their order through the 'My Orders' section after the order has been shipped.
Result 3
Score: 0.548
Topic: delayed_order
Content: Customers should first check the tracking information. If the order remains delayed, they can contact NovaCart support.

QUESTION: I changed my mind. Can I stop my order?
Result 1
Score: 0.535
Topic: cancellation
Content: Orders can be cancelled before they are shipped.
Result 2
Score: 0.494
Topic: address_change
Content: Address changes are possible only before the order has been shipped.
Result 3
Score: 0.432
Topic: tracking
Content: Customers can track their order through the 'My Orders' section after the order has been shipped.


In [64]:
demo = gr.ChatInterface(
    fn=chat_with_support,
    title="🛒 NovaCart Customer Support Assistant",
    description=(
        "Welcome to NovaCart Support! "
        "Ask questions about your account, orders, payments, or other support topics."
    ),
    examples=[
        "I forgot my password.",
        "My payment failed. What should I do?",
        "How can I contact NovaCart support?",
        "How do I track my order?"
    ],
    textbox=gr.Textbox(
        placeholder="Type your question here...",
        container=True
    )
)

demo.launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://bf90a39103c982a06c.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7861 <> https://bf90a39103c982a06c.gradio.live
